# 第3幕『中を覗く』 — 解釈性を自分の手で

> **シリーズの物語** 「小さなモデルを、三つの角度から解剖する」
>
> 同じ小型モデル（Qwen2.5-0.5B など）を主役にして、3冊を通して一本の筋で学びます。
>
> 1. **第1幕『縮める』** — 量子化。重みの精度を落とすと、サイズ・速度・賢さがどう動くかを測る。
> 2. **第2幕『方向を知る』** — abliteration 論文の読解。「拒否は1つの方向に乗っている」という主張を、概念として理解する（コードで外す作業はしない）。
> 3. **第3幕『中を覗く』** — 解釈性。第2幕の土台である「内部表現を読む」技術を、安全な題材（事実の想起）で自分の手で動かす。
>
> 縮めて速くし、その中で何が起きているかを知り、最後に内部を直接観察する——という流れです。


## このノートで動かすこと

第2幕で学んだ道具——**内部状態を読む / 方向を足す / 因果を確かめる**——を、
**安全な題材**で実際に動かします。題材は王道の **事実の想起（factual recall）**:

> `"The Eiffel Tower is in the city of ___"` → モデルはどの層で「Paris」を思い出すのか？

使う技術は2つ、どちらも解釈性の基本道具です。
1. **Logit Lens** — 各層の途中状態を無理やり出力語彙に射影し、「今どの単語を考えているか」を層ごとに覗く。
2. **Activation Patching** — ある実行の中間状態を別の実行に移植し、「どこが答えを決めているか」を**因果的に**特定する。

第2幕の difference-in-means と同じ発想（内部ベクトルを操作して振る舞いの変化を見る）を、無害タスクで体験します。


## 環境

> これも**あなたのPC**で動かす前提です。小さいモデルなので CPU でも可。
```bash
pip install transformer_lens torch matplotlib
```
`transformer_lens` は各層の状態を取り出し・差し替えできる解釈性用ライブラリです。


In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from transformer_lens import HookedTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# 解釈性研究で定番の小型モデル。素直で観察しやすい
model = HookedTransformer.from_pretrained("gpt2", device=DEVICE)
model.eval()
print("layers:", model.cfg.n_layers, " d_model:", model.cfg.d_model)


## パート1 — Logit Lens: 層ごとに『今考えている単語』を覗く

In [ ]:
prompt = "The Eiffel Tower is in the city of"
tokens = model.to_tokens(prompt)

# run_with_cache で全層の残差ストリームを記録
logits, cache = model.run_with_cache(tokens)

# 最終位置の、各層出力(resid_post)を最終LayerNorm+Unembedで語彙に射影する = Logit Lens
answer = " Paris"
ans_id = model.to_single_token(answer)

ranks, probs = [], []
for layer in range(model.cfg.n_layers):
    resid = cache["resid_post", layer][0, -1]          # (d_model,)
    normed = model.ln_final(resid)
    lg = normed @ model.W_U + model.b_U                 # (vocab,)
    p = torch.softmax(lg, -1)
    probs.append(p[ans_id].item())
    ranks.append((lg.argsort(descending=True) == ans_id).nonzero().item())

for l,(r,p) in enumerate(zip(ranks,probs)):
    bar = "#"*int(40*p)
    print(f"L{l:2d}  P(Paris)={p:5.2f}  rank={r:5d} {bar}")


上のバーを見ると、**浅い層では Paris はまだ低確率**で、**ある層から急に浮かび上がる**はずです。
「事実はモデルの特定の深さで形になる」という感覚がつかめます。グラフにします。


In [ ]:
plt.figure(figsize=(7,4))
plt.plot(range(model.cfg.n_layers), probs, "o-")
plt.xlabel("layer"); plt.ylabel("P(' Paris') at final position")
plt.title("Logit Lens: 層が進むにつれ答えが立ち上がる")
plt.grid(alpha=.3); plt.tight_layout(); plt.show()


## パート2 — Activation Patching: 答えを決めているのはどこか（因果）

やり方（第2幕の「足す/消す」と同じ因果の発想）:
- **clean**: `"The Eiffel Tower is in the city of"` → 正解 Paris
- **corrupt**: `"The Colosseum is in the city of"` → 正解 Rome

corrupt を実行しつつ、**clean の中間状態を1か所だけ移植**する。
移植した途端に答えが Paris 寄りに戻ったら、**そこが答えを運んでいた**と分かる。
全(層 × 位置)で試してヒートマップにします。


In [ ]:
clean   = model.to_tokens("The Eiffel Tower is in the city of")
corrupt = model.to_tokens("The Colosseum is   in the city of")
# トークン長を合わせる（ズレる場合はプロンプトを調整）
assert clean.shape == corrupt.shape, (clean.shape, corrupt.shape)

paris = model.to_single_token(" Paris")
rome  = model.to_single_token(" Rome")

_, clean_cache = model.run_with_cache(clean)

def metric(logits):
    # Paris と Rome のロジット差。大きいほど「Paris と答えている」
    last = logits[0, -1]
    return (last[paris] - last[rome]).item()

base_corrupt = metric(model(corrupt))
base_clean   = metric(model(clean))
print(f"clean diff={base_clean:.2f}  corrupt diff={base_corrupt:.2f}")

n_layer = model.cfg.n_layers
seq = clean.shape[1]
heat = np.zeros((n_layer, seq))

def make_hook(layer, pos):
    def hook(resid, hook):
        resid[0, pos] = clean_cache["resid_post", layer][0, pos]  # cleanの状態を移植
        return resid
    return hook

for layer in range(n_layer):
    for pos in range(seq):
        logits = model.run_with_hooks(
            corrupt,
            fwd_hooks=[(f"blocks.{layer}.hook_resid_post", make_hook(layer,pos))])
        # 回復度: corrupt から clean 方向へどれだけ戻ったか [0,1]
        heat[layer,pos] = (metric(logits)-base_corrupt)/(base_clean-base_corrupt+1e-9)


In [ ]:
labels = model.to_str_tokens(clean)
plt.figure(figsize=(10,5))
plt.imshow(heat, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(label="答えの回復度 (1 = cleanに戻った)")
plt.yticks(range(n_layer)); plt.ylabel("layer")
plt.xticks(range(seq), labels, rotation=45, ha="right"); plt.xlabel("token position")
plt.title("Activation Patching: どの(層,位置)が答えを運んでいるか")
plt.tight_layout(); plt.show()


赤く光る (層, 位置) が、**答えを因果的に運んでいる場所**です。
典型的には「主語の最後のトークン」や「最終位置」の特定の層帯が光り、
*情報が主語からコピーされ、後段で読み出される* という流れが見えてきます。

これはまさに第2幕の論文がやったこと——**内部状態を差し替えて振る舞いの変化を測る**——の
無害版です。同じ顕微鏡を、違う（安全な）対象に向けただけ、と分かります。


## パート3 — difference-in-means を安全な軸で体験する

第2幕の「方向を作って足す」を、拒否ではなく **『大文字で書く』スタイル** のような無害な軸で試せます。
考え方の練習として:

1. 「スタイルA」のプロンプト群と「スタイルB」の群を通す
2. 残差の平均差 $d$ を作る
3. 生成中に $d$ を足して、スタイルが寄るか観察する

拒否方向と数式はまったく同じですが、**対象が無害**なので安心して因果の感覚を養えます。
（実装はパート1・2の道具の組み合わせでそのまま書けます。まずは上の2つを動かしてからどうぞ。）


## シリーズのまとめ

- **第1幕『縮める』**: 量子化はサイズと質のトレードオフ。数値で測れば怖くない。
- **第2幕『方向を知る』**: 振る舞い（拒否）はしばしば残差空間の**1方向**に乗る——という経験的発見と、その限界。
- **第3幕『中を覗く』**: Logit Lens と Activation Patching で、**事実がどの層・どこで形になるか**を自分の手で観察した。

一本の筋: **モデルを軽くし(1)、中で意味がどう表されるかを知り(2)、その内部を直接読む顕微鏡を手に入れる(3)。**

### 次に進むなら
- `transformer_lens` の公式チュートリアル（ARENA, Neel Nanda）
- Logit Lens / Activation Patching の原典、attribution patching への発展
- 量子化側: GGUF の k-quant / imatrix、AWQ・GPTQ の仕組み

> 解釈性の道具は強力です。**何を観察するか**の選び方も含めて、学びを楽しんでください。
